# Cookie Cutter — anyui / ipywidgets layout sketch

This notebook is a **layout twin** of `cutter_anyui.html`.

In the browser app the tree is built in `cutter_anyui_main.js` with anyui widgets:

```
VBox
  HBox  toolbar (Shape, Scale, Fit, Insert, Delete, Export)
  HBox  stage
    CurveEditorWidget
    WebGLCutterWidget
  PathTableWidget
```

Here we recreate the **chrome** with ipywidgets. The two canvases still need the JS anywidgets (`es6/curve-editor-widget.js`, `es6/webgl-cutter-widget.js`) pointed at by a thin `anywidget.AnyWidget` subclass — that wiring is the next packaging step, not required to read this notebook.

`turtlePath` is `[[length, angleDegrees], ...]`.

In [1]:
import json
from pathlib import Path

import ipywidgets as W

from IPython.display import display

# Same outlines the ES6 app loads from cookiecutters.js
OUTLINES = {
    "Duck": {
        "name": "Duck",
        "startPoint": [0, 0],
        "startAngle": 180,
        "turtlePath": [
            [0.4, -10], [13.297, 25], [3, -80], [4, 160],
            [22.913, 90], [15, 90], [5, -90], [5, 20],
            [3, 170], [2, -20], [3, -90], [15, 220], [5, -125],
        ],
    }
}

shape = W.Dropdown(options=["Duck", "Heart", "Star", "Blank"], value="Duck", description="Shape")
scale = W.FloatText(value=11.0, description="Scale")
btn_fit = W.Button(description="Fit")
btn_insert = W.Button(description="Insert")
btn_delete = W.Button(description="Delete")
btn_export = W.Button(description="Export JSON", button_style="primary")
status = W.HTML("<em>ipywidgets chrome — hook CurveEditor / WebGL anywidgets here</em>")
table = W.Output()

toolbar = W.HBox([shape, scale, btn_fit, btn_insert, btn_delete, btn_export])
stage = W.HBox([W.HTML("<div style='min-height:220px;flex:1'>Path canvas</div>"),
                W.HTML("<div style='min-height:220px;flex:1'>3D blade</div>")])
root = W.VBox([toolbar, stage, status, table])

def show_table(outline):
    table.clear_output()
    rows = "".join(
        f"<tr><td>{i}</td><td>{seg[0]}</td><td>{seg[1]}</td></tr>"
        for i, seg in enumerate(outline["turtlePath"], start=1)
    )
    with table:
        display(W.HTML(
            "<table><tr><th>#</th><th>Length</th><th>Angle</th></tr>"
            + rows + "</table>"
        ))

def on_shape(change):
    outline = OUTLINES.get(change["new"], {"name": "Custom", "startPoint": [0, 0], "startAngle": 0, "turtlePath": []})
    show_table(outline)
    status.value = f"<code>{json.dumps(outline['turtlePath'][:3])} …</code>"

shape.observe(on_shape, names="value")
btn_export.on_click(lambda *_: print(json.dumps(OUTLINES.get(shape.value, {}), indent=2)))
show_table(OUTLINES["Duck"])
root

## Next packaging step

Point anywidget `_esm` at:

- `es6/curve-editor-widget.js` — keys `turtlePath`, `startPoint`, `startAngle`, `name`, `selected_index`
- `es6/webgl-cutter-widget.js` — plus `outlineScale`, `bladeScale`, `animate`

Then replace the two HTML placeholders with those widgets and keep this same `VBox` / `HBox` tree — that is the dual-environment goal of anyui.